In [ ]:
"""
Requirements:
    pip install opencv-python face_recognition numpy pillow
    (face_recognition requires cmake and dlib — see README)
"""

import cv2
import os
import numpy as np

# face_recognition is optional; detection still works without it
try:
    import face_recognition
    RECOGNITION_AVAILABLE = True
except ImportError:
    RECOGNITION_AVAILABLE = False
    print("⚠️  face_recognition not installed. Running detection-only mode.")

# ---------------------------------------------------------------------------
# Haar Cascade Face Detector
# ---------------------------------------------------------------------------

HAAR_CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(HAAR_CASCADE_PATH)

def detect_faces_haar(gray_frame):
    """Detect faces using Haar Cascade. Returns list of (x, y, w, h)."""
    faces = face_cascade.detectMultiScale(
        gray_frame,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )
    return faces if len(faces) else []

# ---------------------------------------------------------------------------
# Known Faces Database (for recognition)
# ---------------------------------------------------------------------------

class FaceDatabase:
    def __init__(self):
        self.known_encodings = []
        self.known_names = []

    def add_face_from_file(self, image_path: str, name: str):
        if not RECOGNITION_AVAILABLE:
            print("face_recognition not available.")
            return
        if not os.path.exists(image_path):
            print(f"File not found: {image_path}")
            return
        img = face_recognition.load_image_file(image_path)
        encs = face_recognition.face_encodings(img)
        if encs:
            self.known_encodings.append(encs[0])
            self.known_names.append(name)
            print(f"✅ Added {name} to the database.")
        else:
            print(f"⚠️  No face found in {image_path}")

    def identify(self, face_encoding):
        if not self.known_encodings:
            return "Unknown"
        distances = face_recognition.face_distance(self.known_encodings, face_encoding)
        best_idx = np.argmin(distances)
        if distances[best_idx] < 0.5:  # Threshold
            return self.known_names[best_idx]
        return "Unknown"

# ---------------------------------------------------------------------------
# Image Detection
# ---------------------------------------------------------------------------

def detect_in_image(image_path: str, db: FaceDatabase = None, output_path: str = None):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image: {image_path}")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detect_faces_haar(gray)
    print(f"Detected {len(faces)} face(s).")

    for (x, y, w, h) in faces:
        label = "Face"

        if RECOGNITION_AVAILABLE and db:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            locs = [(y, x + w, y + h, x)]
            encs = face_recognition.face_encodings(rgb, locs)
            if encs:
                label = db.identify(encs[0])

        color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        cv2.putText(img, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    if output_path:
        cv2.imwrite(output_path, img)
        print(f"Saved result to {output_path}")
    else:
        cv2.imshow("Face Detection", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

# ---------------------------------------------------------------------------
# Webcam Detection (Live)
# ---------------------------------------------------------------------------

def detect_from_webcam(db: FaceDatabase = None):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Could not open webcam.")
        return

    print("Webcam started. Press 'q' to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detect_faces_haar(gray)

        for (x, y, w, h) in faces:
            label = "Face"

            if RECOGNITION_AVAILABLE and db:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                locs = [(y, x + w, y + h, x)]
                encs = face_recognition.face_encodings(rgb, locs)
                if encs:
                    label = db.identify(encs[0])

            color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        face_count = len(faces)
        cv2.putText(frame, f"Faces: {face_count}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
        cv2.imshow("Face Detection - Press Q to quit", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# ---------------------------------------------------------------------------
# Main CLI
# ---------------------------------------------------------------------------

def main():
    print("=" * 50)
    print("   CodSoft Face Detection & Recognition")
    print("=" * 50)

    db = FaceDatabase()

    while True:
        print("\nOptions:")
        print("  1 → Detect faces in an image file")
        print("  2 → Live webcam face detection")
        if RECOGNITION_AVAILABLE:
            print("  3 → Add known face to database (for recognition)")
        print("  4 → Exit")

        choice = input("\nYour choice: ").strip()

        if choice == "1":
            path = input("Image path: ").strip()
            out = input("Output path (leave blank to display): ").strip() or None
            detect_in_image(path, db=db, output_path=out)

        elif choice == "2":
            detect_from_webcam(db=db)

        elif choice == "3" and RECOGNITION_AVAILABLE:
            path = input("Image path of the person: ").strip()
            name = input("Person's name: ").strip()
            db.add_face_from_file(path, name)

        elif choice == "4":
            print("Goodbye! 👋")
            break

        else:
            print("Invalid choice.")

if __name__ == "__main__":
    main()